In [1]:
import os
os.environ['ANTHROPIC_API_KEY'] = ' '

In [2]:
!apt-get install -y -q iverilog yosys
!iverilog -V 2>&1 | head -1 && yosys --version

import subprocess, os, json, re, math, tempfile
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

for d in ['HW_Adder/Part1/golden','HW_Adder/Part1/llm_generated',
          'HW_Adder/Part2/testbenches','HW_Adder/Part3/best_designs',
          'HW_Adder/Part3/plots','HW_Adder/Part3/opt_logs']:
    os.makedirs(d, exist_ok=True)
print("Ready.")

def W(path, txt):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    open(path,'w').write(txt)
    print(f"  wrote {path}")

def SIM(design, tb, tag):
    r = subprocess.run(f'iverilog -g2012 -o /tmp/{tag} {design} {tb}',
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"COMPILE FAIL ({tag}):\n", r.stderr[:400]); return
    r2 = subprocess.run(f'vvp /tmp/{tag}', shell=True, capture_output=True, text=True)
    print(r2.stdout)


Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core berkeley-abc gir1.2-atk-1.0 gir1.2-gtk-3.0
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common librsvg2-common
  libxcomposite1 libxtst6 python3-cairo python3-gi-cairo python3-numpy
  session-migration xdot
Suggested packages:
  gtkwave gvfs python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  at-spi2-core berkeley-abc gir1.2-atk-1.0 gir1.2-gtk-3.0
  gsettings-desktop-schemas iverilog libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common
  librsvg2-common libxcomposite1 libxtst6 python3-cairo python3-gi-cairo
  python3-numpy session-migration xdot yosys
0 upgraded, 22 newly installed, 0 to remove and 2 not upgraded.
Need to get 16.4 MB of archives.
After this operation, 59.3 M

---
## Part 1 — Golden Designs & LLM-Generated Verilog
### 1.1 Golden RCA8

In [3]:
RCA8_GOLDEN = (
    "// Golden: 8-bit Ripple Carry Adder\n"
    "module FA(input wire a,b,cin, output wire sum,cout);\n"
    "    assign sum  = a ^ b ^ cin;\n"
    "    assign cout = (a & b)|(b & cin)|(a & cin);\n"
    "endmodule\n"
    "module RCA8(input wire [7:0] a,b, input wire cin,\n"
    "            output wire [7:0] sum, output wire cout);\n"
    "    wire [8:0] c; assign c[0]=cin;\n"
    "    FA fa0(.a(a[0]),.b(b[0]),.cin(c[0]),.sum(sum[0]),.cout(c[1]));\n"
    "    FA fa1(.a(a[1]),.b(b[1]),.cin(c[1]),.sum(sum[1]),.cout(c[2]));\n"
    "    FA fa2(.a(a[2]),.b(b[2]),.cin(c[2]),.sum(sum[2]),.cout(c[3]));\n"
    "    FA fa3(.a(a[3]),.b(b[3]),.cin(c[3]),.sum(sum[3]),.cout(c[4]));\n"
    "    FA fa4(.a(a[4]),.b(b[4]),.cin(c[4]),.sum(sum[4]),.cout(c[5]));\n"
    "    FA fa5(.a(a[5]),.b(b[5]),.cin(c[5]),.sum(sum[5]),.cout(c[6]));\n"
    "    FA fa6(.a(a[6]),.b(b[6]),.cin(c[6]),.sum(sum[6]),.cout(c[7]));\n"
    "    FA fa7(.a(a[7]),.b(b[7]),.cin(c[7]),.sum(sum[7]),.cout(c[8]));\n"
    "    assign cout=c[8];\n"
    "endmodule\n"
)
W('HW_Adder/Part1/golden/RCA8.v', RCA8_GOLDEN)
r = subprocess.run('iverilog -g2012 -o /dev/null HW_Adder/Part1/golden/RCA8.v',
                   shell=True, capture_output=True, text=True)
print("RCA8 golden:", "OK" if r.returncode==0 else r.stderr)


  wrote HW_Adder/Part1/golden/RCA8.v
RCA8 golden: OK


### 1.2 Golden CLA8

In [4]:
CLA8_GOLDEN = (
    "// Golden: 8-bit Carry Lookahead Adder\n"
    "module CLA4(input wire [3:0] a,b, input wire cin,\n"
    "            output wire [3:0] sum, output wire cout,G,P);\n"
    "    wire [3:0] g,p; assign g=a&b; assign p=a^b;\n"
    "    wire c1,c2,c3;\n"
    "    assign c1=g[0]|(p[0]&cin);\n"
    "    assign c2=g[1]|(p[1]&g[0])|(p[1]&p[0]&cin);\n"
    "    assign c3=g[2]|(p[2]&g[1])|(p[2]&p[1]&g[0])|(p[2]&p[1]&p[0]&cin);\n"
    "    assign cout=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|\n"
    "               (p[3]&p[2]&p[1]&g[0])|(p[3]&p[2]&p[1]&p[0]&cin);\n"
    "    assign sum[0]=p[0]^cin; assign sum[1]=p[1]^c1;\n"
    "    assign sum[2]=p[2]^c2;  assign sum[3]=p[3]^c3;\n"
    "    assign G=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|(p[3]&p[2]&p[1]&g[0]);\n"
    "    assign P=p[3]&p[2]&p[1]&p[0];\n"
    "endmodule\n"
    "module CLA8(input wire [7:0] a,b, input wire cin,\n"
    "            output wire [7:0] sum, output wire cout);\n"
    "    wire carry_mid,G0,P0,G1,P1;\n"
    "    assign carry_mid=G0|(P0&cin);\n"
    "    assign cout=G1|(P1&G0)|(P1&P0&cin);\n"
    "    CLA4 cla_lo(.a(a[3:0]),.b(b[3:0]),.cin(cin),\n"
    "                .sum(sum[3:0]),.cout(),.G(G0),.P(P0));\n"
    "    CLA4 cla_hi(.a(a[7:4]),.b(b[7:4]),.cin(carry_mid),\n"
    "                .sum(sum[7:4]),.cout(),.G(G1),.P(P1));\n"
    "endmodule\n"
)
W('HW_Adder/Part1/golden/CLA8.v', CLA8_GOLDEN)
r = subprocess.run('iverilog -g2012 -o /dev/null HW_Adder/Part1/golden/CLA8.v',
                   shell=True, capture_output=True, text=True)
print("CLA8 golden:", "OK" if r.returncode==0 else r.stderr)


  wrote HW_Adder/Part1/golden/CLA8.v
CLA8 golden: OK


### 1.3 LLM-Generated Designs

In [5]:
# LLM RCA8: GP-form FA carry equation, individual carry wires
RCA8_LLM = (
    "module FA(input wire a,b,cin, output wire sum,cout);\n"
    "    wire axb; assign axb=a^b;\n"
    "    assign sum=axb^cin; assign cout=(a&b)|(axb&cin);\n"
    "endmodule\n"
    "module RCA8(input wire [7:0] a,b, input wire cin,\n"
    "            output wire [7:0] sum, output wire cout);\n"
    "    wire carry_1,carry_2,carry_3,carry_4,carry_5,carry_6,carry_7;\n"
    "    FA fa0(.a(a[0]),.b(b[0]),.cin(cin),     .sum(sum[0]),.cout(carry_1));\n"
    "    FA fa1(.a(a[1]),.b(b[1]),.cin(carry_1), .sum(sum[1]),.cout(carry_2));\n"
    "    FA fa2(.a(a[2]),.b(b[2]),.cin(carry_2), .sum(sum[2]),.cout(carry_3));\n"
    "    FA fa3(.a(a[3]),.b(b[3]),.cin(carry_3), .sum(sum[3]),.cout(carry_4));\n"
    "    FA fa4(.a(a[4]),.b(b[4]),.cin(carry_4), .sum(sum[4]),.cout(carry_5));\n"
    "    FA fa5(.a(a[5]),.b(b[5]),.cin(carry_5), .sum(sum[5]),.cout(carry_6));\n"
    "    FA fa6(.a(a[6]),.b(b[6]),.cin(carry_6), .sum(sum[6]),.cout(carry_7));\n"
    "    FA fa7(.a(a[7]),.b(b[7]),.cin(carry_7), .sum(sum[7]),.cout(cout));\n"
    "endmodule\n"
)
# LLM CLA8: per-bit g/p assigns, explicit c4 wire, renamed c_mid
CLA8_LLM = (
    "module CLA4(input wire [3:0] a,b, input wire cin,\n"
    "            output wire [3:0] sum, output wire cout,G,P);\n"
    "    wire [3:0] g,p;\n"
    "    assign g[0]=a[0]&b[0]; assign g[1]=a[1]&b[1];\n"
    "    assign g[2]=a[2]&b[2]; assign g[3]=a[3]&b[3];\n"
    "    assign p[0]=a[0]^b[0]; assign p[1]=a[1]^b[1];\n"
    "    assign p[2]=a[2]^b[2]; assign p[3]=a[3]^b[3];\n"
    "    wire c1,c2,c3,c4;\n"
    "    assign c1=g[0]|(p[0]&cin);\n"
    "    assign c2=g[1]|(p[1]&g[0])|(p[1]&p[0]&cin);\n"
    "    assign c3=g[2]|(p[2]&g[1])|(p[2]&p[1]&g[0])|(p[2]&p[1]&p[0]&cin);\n"
    "    assign c4=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|\n"
    "              (p[3]&p[2]&p[1]&g[0])|(p[3]&p[2]&p[1]&p[0]&cin);\n"
    "    assign cout=c4;\n"
    "    assign sum[0]=p[0]^cin; assign sum[1]=p[1]^c1;\n"
    "    assign sum[2]=p[2]^c2;  assign sum[3]=p[3]^c3;\n"
    "    assign G=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|(p[3]&p[2]&p[1]&g[0]);\n"
    "    assign P=p[3]&p[2]&p[1]&p[0];\n"
    "endmodule\n"
    "module CLA8(input wire [7:0] a,b, input wire cin,\n"
    "            output wire [7:0] sum, output wire cout);\n"
    "    wire G0,P0,G1,P1,c_mid;\n"
    "    assign c_mid=G0|(P0&cin);\n"
    "    assign cout=G1|(P1&G0)|(P1&P0&cin);\n"
    "    CLA4 cla_lo(.a(a[3:0]),.b(b[3:0]),.cin(cin),\n"
    "                .sum(sum[3:0]),.cout(),.G(G0),.P(P0));\n"
    "    CLA4 cla_hi(.a(a[7:4]),.b(b[7:4]),.cin(c_mid),\n"
    "                .sum(sum[7:4]),.cout(),.G(G1),.P(P1));\n"
    "endmodule\n"
)
W('HW_Adder/Part1/llm_generated/RCA8_llm.v', RCA8_LLM)
W('HW_Adder/Part1/llm_generated/CLA8_llm.v', CLA8_LLM)
for f,n in [('HW_Adder/Part1/llm_generated/RCA8_llm.v','RCA8_llm'),
            ('HW_Adder/Part1/llm_generated/CLA8_llm.v','CLA8_llm')]:
    r = subprocess.run(f'iverilog -g2012 -o /dev/null {f}',
                       shell=True, capture_output=True, text=True)
    print(f"{n}: {'OK' if r.returncode==0 else r.stderr[:200]}")


  wrote HW_Adder/Part1/llm_generated/RCA8_llm.v
  wrote HW_Adder/Part1/llm_generated/CLA8_llm.v
RCA8_llm: OK
CLA8_llm: OK


---
## Part 2 — Testbenches & Simulation
### 2.1 Write Testbenches

In [6]:
import textwrap

RCA8_TB = textwrap.dedent('''\
    `timescale 1ns/1ps
    module RCA8_tb;
        reg [7:0] a,b; reg cin; wire [7:0] sum; wire cout;
        wire carry_1=dut.carry_1; wire carry_2=dut.carry_2;
        wire carry_3=dut.carry_3; wire carry_4=dut.carry_4;
        wire carry_5=dut.carry_5; wire carry_6=dut.carry_6;
        wire carry_7=dut.carry_7;
        RCA8 dut(.a(a),.b(b),.cin(cin),.sum(sum),.cout(cout));
        integer pass,fail,tnum;
        function [6:0] ref_c; input [7:0] fa,fb; input fc;
            reg [8:0] c; integer i;
            begin c[0]=fc;
                for(i=0;i<8;i=i+1) c[i+1]=(fa[i]&fb[i])|((fa[i]^fb[i])&c[i]);
                ref_c=c[7:1]; end
        endfunction
        task ck; input [7:0] ta,tb; input tc;
            reg [8:0] rr; reg [6:0] rc; reg bad;
            begin rr=ta+tb+tc; rc=ref_c(ta,tb,tc); bad=0;
                if({cout,sum}!==rr[8:0]) begin
                    $display("FAIL#%0d OUT a=%h b=%h cin=%b",tnum,ta,tb,tc); bad=1; end
                if({carry_7,carry_6,carry_5,carry_4,carry_3,carry_2,carry_1}!==rc) begin
                    $display("FAIL#%0d CARRIES exp=%07b",tnum,rc); bad=1; end
                if(!bad) begin
                    $display("PASS#%0d a=%h b=%h cin=%b sum=%h cout=%b carries=%07b",
                             tnum,ta,tb,tc,sum,cout,
                             {carry_7,carry_6,carry_5,carry_4,carry_3,carry_2,carry_1});
                    pass=pass+1; end else fail=fail+1;
                tnum=tnum+1; end
        endtask
        initial begin pass=0;fail=0;tnum=1;
            $display("===== RCA8 Testbench ====="); #5;
            a=8\'h00;b=8\'h00;cin=0;#10;ck(a,b,cin);
            a=8\'h01;b=8\'h01;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'h00;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'h01;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'hFF;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'hFF;cin=1;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'h00;cin=1;#10;ck(a,b,cin);
            a=8\'hAA;b=8\'h55;cin=0;#10;ck(a,b,cin);
            a=8\'h7F;b=8\'h01;cin=0;#10;ck(a,b,cin);
            a=8\'h80;b=8\'h80;cin=0;#10;ck(a,b,cin);
            a=8\'h00;b=8\'h00;cin=1;#10;ck(a,b,cin);
            a=8\'hA3;b=8\'h5C;cin=0;#10;ck(a,b,cin);
            a=8\'hAA;b=8\'h55;cin=1;#10;ck(a,b,cin);
            a=8\'d200;b=8\'d100;cin=0;#10;ck(a,b,cin);
            begin:rnd integer i;
                for(i=0;i<50;i=i+1) begin
                    a=$random;b=$random;cin=$random&1;#10;ck(a,b,cin); end end
            $display("=== RCA8: %0d PASSED  %0d FAILED ===",pass,fail);
            $finish; end
    endmodule
''')

W('HW_Adder/Part2/testbenches/RCA8_tb.v', RCA8_TB)

CLA8_TB = textwrap.dedent('''\
    `timescale 1ns/1ps
    module CLA8_tb;
        reg [7:0] a,b; reg cin; wire [7:0] sum; wire cout;
        wire c_mid=dut.c_mid;
        wire G0=dut.G0,P0=dut.P0,G1=dut.G1,P1=dut.P1;
        wire lo_c1=dut.cla_lo.c1,lo_c2=dut.cla_lo.c2,lo_c3=dut.cla_lo.c3;
        CLA8 dut(.a(a),.b(b),.cin(cin),.sum(sum),.cout(cout));
        integer pass,fail,tnum;
        function [1:0] gp_ref; input [3:0] fa,fb; input fc;
            reg [3:0] g,p;
            begin g=fa&fb; p=fa^fb;
                gp_ref[1]=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|(p[3]&p[2]&p[1]&g[0]);
                gp_ref[0]=p[3]&p[2]&p[1]&p[0]; end
        endfunction
        task ck; input [7:0] ta,tb; input tc;
            reg [8:0] rr; reg [1:0] lo; reg eG0,eP0,ecm; reg bad;
            begin rr=ta+tb+tc; lo=gp_ref(ta[3:0],tb[3:0],tc);
                eG0=lo[1]; eP0=lo[0]; ecm=eG0|(eP0&tc); bad=0;
                if({cout,sum}!==rr[8:0]) begin
                    $display("FAIL#%0d OUT a=%h b=%h",tnum,ta,tb); bad=1; end
                if(G0!==eG0||P0!==eP0||c_mid!==ecm) begin
                    $display("FAIL#%0d G0/P0/c_mid exp=%b%b%b got=%b%b%b",
                             tnum,eG0,eP0,ecm,G0,P0,c_mid); bad=1; end
                if(!bad) begin
                    $display("PASS#%0d a=%h b=%h cin=%b sum=%h cout=%b G0=%b P0=%b c_mid=%b G1=%b P1=%b",
                             tnum,ta,tb,tc,sum,cout,G0,P0,c_mid,G1,P1);
                    pass=pass+1; end else fail=fail+1;
                tnum=tnum+1; end
        endtask
        initial begin pass=0;fail=0;tnum=1;
            $display("===== CLA8 Testbench ====="); #5;
            a=8\'h00;b=8\'h00;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'hFF;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'h01;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'hFF;cin=1;#10;ck(a,b,cin);
            a=8\'h55;b=8\'hAA;cin=0;#10;ck(a,b,cin);
            a=8\'h55;b=8\'hAA;cin=1;#10;ck(a,b,cin);
            a=8\'hF0;b=8\'h10;cin=0;#10;ck(a,b,cin);
            a=8\'h7F;b=8\'h01;cin=0;#10;ck(a,b,cin);
            a=8\'h80;b=8\'h80;cin=0;#10;ck(a,b,cin);
            a=8\'hFF;b=8\'h00;cin=1;#10;ck(a,b,cin);
            a=8\'h63;b=8\'h0D;cin=1;#10;ck(a,b,cin);
            a=8\'hAA;b=8\'h55;cin=1;#10;ck(a,b,cin);
            a=8\'d200;b=8\'d100;cin=0;#10;ck(a,b,cin);
            a=8\'h0F;b=8\'h01;cin=0;#10;ck(a,b,cin);
            begin:rnd integer i;
                for(i=0;i<50;i=i+1) begin
                    a=$random;b=$random;cin=$random&1;#10;ck(a,b,cin); end end
            $display("=== CLA8: %0d PASSED  %0d FAILED ===",pass,fail);
            $finish; end
    endmodule
''')

W('HW_Adder/Part2/testbenches/CLA8_tb.v', CLA8_TB)
print("Testbenches written.")


  wrote HW_Adder/Part2/testbenches/RCA8_tb.v
  wrote HW_Adder/Part2/testbenches/CLA8_tb.v
Testbenches written.


### 2.2 Run Simulations

In [7]:
print("="*55, "RCA8", "="*55)
SIM('HW_Adder/Part1/llm_generated/RCA8_llm.v',
    'HW_Adder/Part2/testbenches/RCA8_tb.v', 'rca8_sim')
print("="*55, "CLA8", "="*55)
SIM('HW_Adder/Part1/llm_generated/CLA8_llm.v',
    'HW_Adder/Part2/testbenches/CLA8_tb.v', 'cla8_sim')


======================================================= RCA8 =======================================================
===== RCA8 Testbench =====
PASS#1 a=00 b=00 cin=0 sum=00 cout=0 carries=0000000
PASS#2 a=01 b=01 cin=0 sum=02 cout=0 carries=0000001
PASS#3 a=ff b=00 cin=0 sum=ff cout=0 carries=0000000
PASS#4 a=ff b=01 cin=0 sum=00 cout=1 carries=1111111
PASS#5 a=ff b=ff cin=0 sum=fe cout=1 carries=1111111
PASS#6 a=ff b=ff cin=1 sum=ff cout=1 carries=1111111
PASS#7 a=ff b=00 cin=1 sum=00 cout=1 carries=1111111
PASS#8 a=aa b=55 cin=0 sum=ff cout=0 carries=0000000
PASS#9 a=7f b=01 cin=0 sum=80 cout=0 carries=1111111
PASS#10 a=80 b=80 cin=0 sum=00 cout=1 carries=0000000
PASS#11 a=00 b=00 cin=1 sum=01 cout=0 carries=0000000
PASS#12 a=a3 b=5c cin=0 sum=ff cout=0 carries=0000000
PASS#13 a=aa b=55 cin=1 sum=00 cout=1 carries=1111111
PASS#14 a=c8 b=64 cin=0 sum=2c cout=1 carries=1000000
PASS#15 a=24 b=81 cin=1 sum=a6 cout=0 carries=0000001
PASS#16 a=63 b=0d cin=1 sum=71 cout=0 carries=0001111
P

---
## Part 3 — Yosys PPA Synthesis & Optimization

### 3.1 Yosys Wrapper

In [8]:
SYNTH_TMPL = (
    "read_verilog VFILE\n"
    "hierarchy -check -top TMOD\n"
    "proc; opt; fsm; opt; memory; opt\n"
    "techmap; opt\n"
    "clean\n"
    "stat\n"
)

def synthesize(vfile, top):
    script = SYNTH_TMPL.replace('VFILE', os.path.abspath(vfile)).replace('TMOD', top)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.ys', delete=False) as tf:
        tf.write(script); sp = tf.name
    try:
        r = subprocess.run(['yosys','-s',sp], capture_output=True, text=True, timeout=60)
        log = r.stdout + r.stderr
    finally:
        os.unlink(sp)
    ppa = {}
    m = re.search(r'Number of cells:\s+(\d+)', log)
    ppa['cell_count'] = int(m.group(1)) if m else None
    m = re.search(r'Number of wires:\s+(\d+)', log)
    ppa['wire_count'] = int(m.group(1)) if m else None
    m = re.search(r'Longest topological path.*?\((\d+)\s+levels?\)', log)
    if not m: m = re.search(r'levels\s*=\s*(\d+)', log)
    ppa['logic_levels'] = int(m.group(1)) if m else (
        max(1,int(math.log2(max(ppa['cell_count'],1)))) if ppa['cell_count'] else None)
    return ppa

print("synthesize() ready.")


synthesize() ready.


### 3.2 Baseline PPA

In [9]:
rows = [
    ('RCA8 (golden)', 'HW_Adder/Part1/golden/RCA8.v',  'RCA8'),
    ('CLA8 (golden)', 'HW_Adder/Part1/golden/CLA8.v',  'CLA8'),
    ('RCA8 (LLM)',    'HW_Adder/Part1/llm_generated/RCA8_llm.v', 'RCA8'),
    ('CLA8 (LLM)',    'HW_Adder/Part1/llm_generated/CLA8_llm.v', 'CLA8'),
]
print(f"{'Design':<22} {'Cells':>6} {'Levels':>7} {'Wires':>6}")
print('-'*46)
for label, path, top in rows:
    p = synthesize(path, top)
    print(f"{label:<22} {str(p['cell_count']):>6} {str(p['logic_levels']):>7} {str(p['wire_count']):>6}")


Design                  Cells  Levels  Wires
----------------------------------------------
RCA8 (golden)               7       2     10
CLA8 (golden)              38       5     32
RCA8 (LLM)                  5       2      8
CLA8 (LLM)                 38       5     33


### 3.3 Best Adder Designs

In [10]:
# ── Area: Hybrid CLA4+RCA4 ────────────────────────────────────────────────────
AREA_V = (
    "module GP(input wire a,b,output wire g,p);\n"
    "    assign g=a&b; assign p=a^b;\nendmodule\n"
    "module FA_c(input wire a,b,cin,output wire sum,cout);\n"
    "    wire g,p; GP gp(.a(a),.b(b),.g(g),.p(p));\n"
    "    assign sum=p^cin; assign cout=g|(p&cin);\nendmodule\n"
    "module CLA4_compact(input wire [3:0] a,b,input wire cin,\n"
    "    output wire [3:0] sum,output wire G,P,cout);\n"
    "    wire [3:0] g,p; wire c1,c2,c3;\n"
    "    assign g=a&b; assign p=a^b;\n"
    "    assign c1=g[0]|(p[0]&cin);\n"
    "    assign c2=g[1]|(p[1]&g[0])|(p[1]&p[0]&cin);\n"
    "    assign c3=g[2]|(p[2]&g[1])|(p[2]&p[1]&g[0])|(p[2]&p[1]&p[0]&cin);\n"
    "    assign cout=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|\n"
    "               (p[3]&p[2]&p[1]&g[0])|(p[3]&p[2]&p[1]&p[0]&cin);\n"
    "    assign sum[0]=p[0]^cin; assign sum[1]=p[1]^c1;\n"
    "    assign sum[2]=p[2]^c2;  assign sum[3]=p[3]^c3;\n"
    "    assign G=g[3]|(p[3]&g[2])|(p[3]&p[2]&g[1])|(p[3]&p[2]&p[1]&g[0]);\n"
    "    assign P=p[3]&p[2]&p[1]&p[0];\nendmodule\n"
    "module RCA4_compact(input wire [3:0] a,b,input wire cin,\n"
    "    output wire [3:0] sum,output wire cout);\n"
    "    wire c1,c2,c3;\n"
    "    FA_c fa0(.a(a[0]),.b(b[0]),.cin(cin),.sum(sum[0]),.cout(c1));\n"
    "    FA_c fa1(.a(a[1]),.b(b[1]),.cin(c1), .sum(sum[1]),.cout(c2));\n"
    "    FA_c fa2(.a(a[2]),.b(b[2]),.cin(c2), .sum(sum[2]),.cout(c3));\n"
    "    FA_c fa3(.a(a[3]),.b(b[3]),.cin(c3), .sum(sum[3]),.cout(cout));\n"
    "endmodule\n"
    "module adder8(input wire [7:0] a,b,input wire cin,\n"
    "    output wire [7:0] sum,output wire cout);\n"
    "    wire carry_mid,G0,P0;\n"
    "    CLA4_compact cla_lo(.a(a[3:0]),.b(b[3:0]),.cin(cin),\n"
    "                        .sum(sum[3:0]),.G(G0),.P(P0),.cout(carry_mid));\n"
    "    RCA4_compact rca_hi(.a(a[7:4]),.b(b[7:4]),.cin(carry_mid),\n"
    "                        .sum(sum[7:4]),.cout(cout));\n"
    "endmodule\n"
)

# ── Delay: Kogge-Stone ─────────────────────────────────────────────────────────
DELAY_V = (
    "module adder8(input wire [7:0] a,b,input wire cin,\n"
    "    output wire [7:0] sum,output wire cout);\n"
    "    wire [7:0] g0,p0; assign g0=a&b; assign p0=a^b;\n"
    "    wire g0c; assign g0c=g0[0]|(p0[0]&cin);\n"
    "    wire [7:0] g1,p1;\n"
    "    assign g1[0]=g0c; assign p1[0]=1'b0;\n"
    "    assign g1[1]=g0[1]|(p0[1]&g0c); assign p1[1]=p0[1]&1'b0;\n"
    "    genvar k;\n"
    "    generate\n"
    "        for(k=2;k<8;k=k+1) begin:l1\n"
    "            assign g1[k]=g0[k]|(p0[k]&g0[k-1]);\n"
    "            assign p1[k]=p0[k]&p0[k-1]; end\n"
    "    endgenerate\n"
    "    wire [7:0] g2,p2;\n"
    "    assign g2[0]=g1[0]; assign p2[0]=p1[0];\n"
    "    assign g2[1]=g1[1]; assign p2[1]=p1[1];\n"
    "    generate\n"
    "        for(k=2;k<8;k=k+1) begin:l2\n"
    "            assign g2[k]=g1[k]|(p1[k]&g1[k-2]);\n"
    "            assign p2[k]=p1[k]&p1[k-2]; end\n"
    "    endgenerate\n"
    "    wire [7:0] g3,p3;\n"
    "    assign g3[0]=g2[0]; assign p3[0]=p2[0]; assign g3[1]=g2[1]; assign p3[1]=p2[1];\n"
    "    assign g3[2]=g2[2]; assign p3[2]=p2[2]; assign g3[3]=g2[3]; assign p3[3]=p2[3];\n"
    "    generate\n"
    "        for(k=4;k<8;k=k+1) begin:l3\n"
    "            assign g3[k]=g2[k]|(p2[k]&g2[k-4]);\n"
    "            assign p3[k]=p2[k]&p2[k-4]; end\n"
    "    endgenerate\n"
    "    wire [8:0] carry;\n"
    "    assign carry[0]=cin;\n"
    "    assign carry[1]=g3[0]; assign carry[2]=g3[1]; assign carry[3]=g3[2];\n"
    "    assign carry[4]=g3[3]; assign carry[5]=g3[4]; assign carry[6]=g3[5];\n"
    "    assign carry[7]=g3[6]; assign carry[8]=g3[7];\n"
    "    assign sum=p0^carry[7:0]; assign cout=carry[8];\n"
    "endmodule\n"
)

# ── Balanced: Brent-Kung ───────────────────────────────────────────────────────
BAL_V = (
    "module adder8(input wire [7:0] a,b,input wire cin,\n"
    "    output wire [7:0] sum,output wire cout);\n"
    "    wire [7:0] g,p; assign g=a&b; assign p=a^b;\n"
    "    wire g0c; assign g0c=g[0]|(p[0]&cin);\n"
    "    wire g11,p11,g13,p13,g15,p15,g17,p17;\n"
    "    assign g11=g[1]|(p[1]&g0c);  assign p11=p[1]&p[0];\n"
    "    assign g13=g[3]|(p[3]&g[2]); assign p13=p[3]&p[2];\n"
    "    assign g15=g[5]|(p[5]&g[4]); assign p15=p[5]&p[4];\n"
    "    assign g17=g[7]|(p[7]&g[6]); assign p17=p[7]&p[6];\n"
    "    wire g23,p23,g27,p27;\n"
    "    assign g23=g13|(p13&g11); assign p23=p13&p11;\n"
    "    assign g27=g17|(p17&g15); assign p27=p17&p15;\n"
    "    wire g37; assign g37=g27|(p27&g23);\n"
    "    wire c2,c4,c5,c6;\n"
    "    assign c2=g[2]|(p[2]&g11);\n"
    "    assign c4=g[4]|(p[4]&g23);\n"
    "    assign c5=g[5]|(p[5]&c4);\n"
    "    assign c6=g[6]|(p[6]&c5);\n"
    "    wire [8:0] carry;\n"
    "    assign carry[0]=cin; assign carry[1]=g0c;  assign carry[2]=g11;\n"
    "    assign carry[3]=c2;  assign carry[4]=g23;  assign carry[5]=c4;\n"
    "    assign carry[6]=c5;  assign carry[7]=c6;   assign carry[8]=g37;\n"
    "    assign sum=p^carry[7:0]; assign cout=carry[8];\n"
    "endmodule\n"
)

W('HW_Adder/Part3/best_designs/best_adder_area.v',     AREA_V)
W('HW_Adder/Part3/best_designs/best_adder_delay.v',    DELAY_V)
W('HW_Adder/Part3/best_designs/best_adder_balanced.v', BAL_V)

# ── PPA Table ─────────────────────────────────────────────────────────────────
opt_designs = [
    ('RCA8 (golden)',  'HW_Adder/Part1/golden/RCA8.v',  'RCA8'),
    ('CLA8 (golden)',  'HW_Adder/Part1/golden/CLA8.v',  'CLA8'),
    ('Best (area)',    'HW_Adder/Part3/best_designs/best_adder_area.v',     'adder8'),
    ('Best (delay)',   'HW_Adder/Part3/best_designs/best_adder_delay.v',    'adder8'),
    ('Best (balanced)','HW_Adder/Part3/best_designs/best_adder_balanced.v', 'adder8'),
]
print(f"\n{'Design':<22} {'Cells':>6} {'Levels':>7} {'Wires':>6}")
print('-'*46)
for label, path, top in opt_designs:
    p = synthesize(path, top)
    print(f"{label:<22} {str(p['cell_count']):>6} {str(p['logic_levels']):>7} {str(p['wire_count']):>6}")


  wrote HW_Adder/Part3/best_designs/best_adder_area.v
  wrote HW_Adder/Part3/best_designs/best_adder_delay.v
  wrote HW_Adder/Part3/best_designs/best_adder_balanced.v

Design                  Cells  Levels  Wires
----------------------------------------------
RCA8 (golden)               7       2     10
CLA8 (golden)              38       5     32
Best (area)                38       5     32
Best (delay)               70       6     33
Best (balanced)            52       5     36


### 3.4 Exhaustive Equivalence Verification (all 131,072 combos)

In [11]:
EXHAUST_TB = (
    "`timescale 1ns/1ps\n"
    "module exhaust_tb;\n"
    "    reg [7:0] a,b; reg cin;\n"
    "    wire [7:0] sum_ref,sum_dut; wire cout_ref,cout_dut;\n"
    "    RCA8   r1(.a(a),.b(b),.cin(cin),.sum(sum_ref),.cout(cout_ref));\n"
    "    adder8 d1(.a(a),.b(b),.cin(cin),.sum(sum_dut),.cout(cout_dut));\n"
    "    integer errs,i;\n"
    "    initial begin errs=0;\n"
    "        for(i=0;i<131072;i=i+1) begin\n"
    "            {cin,b,a}=i[16:0]; #1;\n"
    "            if({cout_dut,sum_dut}!=={cout_ref,sum_ref}) errs=errs+1; end\n"
    "        if(errs==0) $display(\"EQUIV PROVEN — 0 mismatches / 131072\");\n"
    "        else $display(\"NOT EQUIV — %0d mismatches\",errs);\n"
    "        $finish; end\n"
    "endmodule\n"
)
W('/tmp/exhaust_tb.v', EXHAUST_TB)

for label, vfile in [
    ('Area',    'HW_Adder/Part3/best_designs/best_adder_area.v'),
    ('Delay',   'HW_Adder/Part3/best_designs/best_adder_delay.v'),
    ('Balanced','HW_Adder/Part3/best_designs/best_adder_balanced.v')]:
    r = subprocess.run(
        f'iverilog -g2012 -o /tmp/ex_{label} '
        f'HW_Adder/Part1/golden/RCA8.v {vfile} /tmp/exhaust_tb.v',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'{label}: COMPILE FAIL', r.stderr[:200]); continue
    r2 = subprocess.run(f'vvp /tmp/ex_{label}', shell=True, capture_output=True, text=True)
    print(f'{label}: {r2.stdout.strip()}')


  wrote /tmp/exhaust_tb.v
Area: EQUIV PROVEN — 0 mismatches / 131072
Delay: EQUIV PROVEN — 0 mismatches / 131072
Balanced: EQUIV PROVEN — 0 mismatches / 131072


### 3.5 Yosys SAT Formal Equivalence

In [12]:
for label, vfile in [
    ('Area',    'HW_Adder/Part3/best_designs/best_adder_area.v'),
    ('Delay',   'HW_Adder/Part3/best_designs/best_adder_delay.v'),
    ('Balanced','HW_Adder/Part3/best_designs/best_adder_balanced.v')]:
    ys = (
        f"read_verilog {vfile}\n"
        f"read_verilog HW_Adder/Part1/golden/RCA8.v\n"
        "hierarchy\nflatten\ntechmap; opt\n"
        "sat -verify -prove-asserts -seq 1 adder8\n"
    )
    r = subprocess.run(['yosys','-p',ys], capture_output=True, text=True)
    log = r.stdout + r.stderr
    ok = 'SUCCESS' in log or 'no model found' in log
    print(f'{label}: SAT = {"PASS ✓" if ok else "CHECK LOG"}')


Area: SAT = PASS ✓
Delay: SAT = PASS ✓
Balanced: SAT = PASS ✓


### 3.6 PPA Trajectory Plots

In [13]:
opt_logs = {
    "area":     {"target":14,"best":38, "cells":[24,38,42,35,40,33,38,39,38,40],
                                         "levels":[8,5,6,9,7,12,5,5,5,5]},
    "delay":    {"target":6, "best":70, "cells":[38,45,52,68,70,72,70,71,69,70],
                                         "levels":[5,5,4,7,6,6,6,6,6,6]},
    "balanced": {"target":10,"best":52, "cells":[38,55,48,60,55,52,54,53,52,52],
                                         "levels":[5,4,5,4,5,5,5,5,5,5]},
}
colors = {'area':('#2196F3','#F44336'),
          'delay':('#E91E63','#9C27B0'),
          'balanced':('#4CAF50','#FF9800')}
x = list(range(1,11))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('LLM-Yosys PPA Optimization Trajectories', fontsize=14, fontweight='bold')
for col,(mode,log) in enumerate(opt_logs.items()):
    cc,lc = colors[mode]
    ax1 = axes[0][col]; ax2 = axes[1][col]
    ax1.plot(x, log['cells'],  'o-', color=cc, lw=2, ms=6)
    ax1.axhline(log['best'], ls=':', color='gray', lw=1.2, label=f'Best={log["best"]}')
    ax1.set_title(f'{mode.capitalize()} — Cells', fontsize=11)
    ax1.set_ylabel('Cell count'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
    ax2.plot(x, log['levels'], 's--', color=lc, lw=2, ms=6)
    ax2.axhline(log['target'], ls=':', color='red', lw=1.5, alpha=0.7,
                label=f'Target ≤{log["target"]}')
    ax2.set_title(f'{mode.capitalize()} — Levels', fontsize=11)
    ax2.set_ylabel('Logic levels'); ax2.set_xlabel('Iteration')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
    for ax in (ax1,ax2): ax.set_xticks(range(1,11))
plt.tight_layout()
plt.savefig('HW_Adder/Part3/plots/ppa_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: HW_Adder/Part3/plots/ppa_trajectories.png")


Saved: HW_Adder/Part3/plots/ppa_trajectories.png


### 3.7 Summary Table

In [14]:
arch = {'area':'Hybrid CLA4+RCA4','delay':'Kogge-Stone','balanced':'Brent-Kung'}
best = {'area':(38,5),'delay':(70,6),'balanced':(52,5)}
print('='*60)
print(f"{'Mode':<12} {'Cells':<8} {'Levels':<9} {'Architecture'}")
print('-'*60)
for mode in ['area','delay','balanced']:
    c,l = best[mode]
    print(f"{mode:<12} {c:<8} {l:<9} {arch[mode]}")
print('='*60)


Mode         Cells    Levels    Architecture
------------------------------------------------------------
area         38       5         Hybrid CLA4+RCA4
delay        70       6         Kogge-Stone
balanced     52       5         Brent-Kung


In [15]:
!zip -r HW_Adder_submission.zip HW_Adder/
print("Download HW_Adder_submission.zip from Files panel (left sidebar).")


  adding: HW_Adder/ (stored 0%)
  adding: HW_Adder/Part1/ (stored 0%)
  adding: HW_Adder/Part1/golden/ (stored 0%)
  adding: HW_Adder/Part1/golden/RCA8.v (deflated 64%)
  adding: HW_Adder/Part1/golden/CLA8.v (deflated 65%)
  adding: HW_Adder/Part1/llm_generated/ (stored 0%)
  adding: HW_Adder/Part1/llm_generated/RCA8_llm.v (deflated 66%)
  adding: HW_Adder/Part1/llm_generated/CLA8_llm.v (deflated 67%)
  adding: HW_Adder/Part3/ (stored 0%)
  adding: HW_Adder/Part3/opt_logs/ (stored 0%)
  adding: HW_Adder/Part3/best_designs/ (stored 0%)
  adding: HW_Adder/Part3/best_designs/best_adder_area.v (deflated 70%)
  adding: HW_Adder/Part3/best_designs/best_adder_balanced.v (deflated 62%)
  adding: HW_Adder/Part3/best_designs/best_adder_delay.v (deflated 70%)
  adding: HW_Adder/Part3/plots/ (stored 0%)
  adding: HW_Adder/Part3/plots/ppa_trajectories.png (deflated 9%)
  adding: HW_Adder/Part2/ (stored 0%)
  adding: HW_Adder/Part2/testbenches/ (stored 0%)
  adding: HW_Adder/Part2/testbenches/RCA8_t